# 3. Heterogeneity Computations
The goal in this notebook is to generate estimates of heterogeneity by computing $I^2$, the proportion between-study variance due to heterogeneity rather than random sampling error. We compute this via simulation. Should be run after the Stan model is run, but before the visuals are generated.

In [12]:
library(tidyverse)
library(ggplot2)
library(rstan)
library(Hmisc)
set.seed(20)

In [13]:
inv_logit = plogis
logit = qlogis

In [14]:
base_dir = getwd()
io_set = "Main Results"
model_input_dir  = file.path(base_dir, "Processed Data", io_set)
model_output_dir = file.path(base_dir, "Model Output", io_set)
output_dir = file.path(base_dir, "Heterogeneity Estimates", io_set)

#sev_class_type = "1997type"
#sev_class_type = "2009type"
sev_class_type = "hospitalisation"
save_output = TRUE

In [15]:
#Read in the model input data (which gives information on outcomes and scenarios)
input_data = file.path(model_input_dir, paste0("data_", sev_class_type, ".rds")) %>% readRDS

#Read in the model results. We are primarily interested in the pooled estimates p and the outcome estimates theta
model_results = file.path(model_output_dir, paste0("Results_LogisticRegression_mean=0_sd=2_sd_mean=0.5_sdsd=2_", sev_class_type, ".rds")) %>% readRDS

In [16]:
#Extract some variables from input data to get correct indices on things
scenario_df = input_data$scenario_df
outcome_df = input_data$outcome_df %>% mutate(ThetaIndex = 1:nrow(.))

#Get the reference region and the non-reference one
ref_region = input_data$ref_region
non_ref_region = ifelse(ref_region == "Asia", "Americas", "Asia")
ref_serotype_exposure = input_data$ref_serotype_exposure #Get reference serotype-prior exposure combination
ref_scenario = input_data$ref_scenario #Get reference scenario
non_ref_seroprior = input_data$char_mat_guide %>% filter(CharMatIndex >0) %>% pull(SeroPriorExp) %>% as.character #Get all non-reference serotype-prior exposures

#Get scenario labels ordered according to the model output which gives the reference scenario, then all non-reference serotype prior exposures in the reference region,
#followed by the reference serotype-exposure in the non-reference region, and the rest of the serotype-prior exposures in the non-reference region
scenario_labels = c(ref_scenario, 
                    paste0(ref_region, "-", non_ref_seroprior),
                    paste0(non_ref_region, "-", c(ref_serotype_exposure, non_ref_seroprior))
                   )

#Add these indices to the scenario DataFrame from the model input
scenario_p_matcher = data.frame(Scenario = scenario_labels) %>% mutate(ScenarioIndex = 1:nrow(.))
scenario_df = scenario_df %>% left_join(scenario_p_matcher, by = "Scenario")
scenario_df

      SeroPriorExp   Region                 Scenario NumStudies Severe NonSevere     N Inclusion ScenIndex CharMatIndex ScenarioIndex
1    Unknown-DENV1 Americas   Americas-Unknown-DENV1          7   7496     17413 24909  Included         1            1            11
2    Unknown-DENV2 Americas   Americas-Unknown-DENV2          6   1326      2165  3491  Included         2            2            12
3    Unknown-DENV3 Americas   Americas-Unknown-DENV3          3   1770      2555  4325  Included         3            3            13
4    Unknown-DENV4 Americas   Americas-Unknown-DENV4          2   2011      6218  8229  Included         4            4            14
5    Primary-DENV1 Americas   Americas-Primary-DENV1          0      0         0     0  Excluded        -1            5            15
6    Primary-DENV2 Americas   Americas-Primary-DENV2          0      0         0     0  Excluded        -1            6            16
7    Primary-DENV3 Americas   Americas-Primary-DENV3          

In [19]:
scenario_p_matcher

                   Scenario ScenarioIndex
1      Asia-Secondary-DENV2             1
2        Asia-Unknown-DENV1             2
3        Asia-Unknown-DENV2             3
4        Asia-Unknown-DENV3             4
5        Asia-Unknown-DENV4             5
6        Asia-Primary-DENV1             6
7        Asia-Primary-DENV2             7
8        Asia-Primary-DENV3             8
9      Asia-Secondary-DENV1             9
10 Americas-Secondary-DENV2            10
11   Americas-Unknown-DENV1            11
12   Americas-Unknown-DENV2            12
13   Americas-Unknown-DENV3            13
14   Americas-Unknown-DENV4            14
15   Americas-Primary-DENV1            15
16   Americas-Primary-DENV2            16
17   Americas-Primary-DENV3            17
18 Americas-Secondary-DENV1            18

In [17]:
#We only generate I^2 estimates for included scenarios (those with at least 2 studies contributing data)
included_scenarios = scenario_df %>% filter(Inclusion == "Included")

#Get the available posterior samples for p and theta
p_est_draws = extract(model_results, pars = "p")$p
theta_est_draws = extract(model_results, pars = "theta")$theta

In [7]:
theta_est_draws

          
iterations       [,1]       [,2]       [,3]       [,4]         [,5]
      [1,] 0.01612830 0.06798049 0.03624964 0.01592230 2.264972e-03
      [2,] 0.01655135 0.06897513 0.03365619 0.01650295 3.097150e-06
      [3,] 0.01557343 0.06636992 0.03141153 0.01910877 1.409407e-05
      [4,] 0.01762640 0.06875691 0.03199285 0.01753710 5.717268e-04
      [5,] 0.01661402 0.07124689 0.02858487 0.01655567 1.429291e-03
      [6,] 0.01523797 0.08058296 0.03203034 0.02026405 8.431128e-04
      [7,] 0.01452343 0.06099415 0.03349409 0.01929324 6.605453e-04
      [8,] 0.01656609 0.06877245 0.03445064 0.01463265 9.791570e-04
          
iterations         [,6]         [,7]         [,8]         [,9]      [,10]
      [1,] 1.799784e-02 0.0105374884 0.0191178063 0.0404380055 0.17493404
      [2,] 5.206973e-03 0.0086246525 0.0243066465 0.0435963114 0.11317748
      [3,] 1.082440e-02 0.0071326794 0.0735486126 0.0421605998 0.10224494
      [4,] 4.382347e-03 0.0053627769 0.0595186699 0.0063566440 0.18131

In [8]:
num_samples = nrow(p_est_draws) # Number of posterior samples
num_included_scenarios = nrow(included_scenarios) #Number of included scenarios to generate I^2 estimates for 
I_2_est_mat = array(-1, dim = c(num_included_scenarios, num_samples)) #Matrix of I^2 estimates (with each posterior sample corresponding to one I^2 estimate)
correction = 0.00001 #To prevent NaN in I2 estimates

#For each row corresponding to an included scenario
for(curr_row_ind in 1:num_included_scenarios){
    curr_row = included_scenarios[curr_row_ind, ]
    curr_scenario = curr_row$Scenario #Name of scenario to retrieve from outcome_df
    curr_scenario_ind = curr_row$ScenarioIndex #Index of Scenario so we know which value of p to get
    curr_outcomes = outcome_df %>% filter(Scenario == curr_scenario) #Get the relevant outcomes that are part of the scenario
    num_outcomes = nrow(curr_outcomes) #Get number of outcomes
    curr_p_vals = p_est_draws[ , curr_scenario_ind] #Get the values of p (column corresponding to the scenario index)
    
    curr_theta_indices = curr_outcomes %>% pull(ThetaIndex) #Get the theta index of each outcome. These should just be from 1:num_outcomes in the order of the outcome_df
    curr_n_vals = curr_outcomes %>% pull(N) #Sample size of each outcome in the given scenario
    curr_theta_vals = theta_est_draws[, curr_theta_indices] #matrix of theta values that match the outcomes for the given scenario
    
    #For each sample, we compute an estimate of I^2 (this means matching indices of p and thetas)
    I_2_est_samples = array(-1, dim = num_samples)
    for(curr_sample_ind in 1:num_samples){
        curr_p = curr_p_vals[curr_sample_ind] #Current p sample from the posterior distribution
        curr_theta = curr_theta_vals[curr_sample_ind, ] #Current theta samples from the posterior distribution
        
        without_het = array(-1, dim = length(num_outcomes)) #Between study variance in the absence of heterogeneity
        with_het = array(-1, dim = length(num_outcomes)) #Between study variance given heterogeneity
        
        for(i in 1:num_outcomes){
            without_het[i] = rbinom(n = 1, size = curr_n_vals[i], prob = curr_p) #For each outcome in the scenario, generate 1 random binomial draw using the outcome's sample size and the pooled estimate

            #We also generate 1 random binomial draw still using the outcome's sample size,
            #but this time using theta as the probability (includes random effect)
            with_het[i] = rbinom(n = 1, size = curr_n_vals[i], prob = curr_theta[i]) 
        }

        #Convert the binomial draws to proportions by dividing by the sample sizes
        without_het_props = without_het / curr_n_vals
        with_het_props = with_het / curr_n_vals

        #Compute the variance between proportions with and without heterogeneity - adding in the correction to both values to prevent divisions by 0 
        #which could occur for scenarios with only a small number of studies and/or smalller sample sizes
        var_het = var(with_het_props) + correction
        var_without_het = var(without_het_props) + correction
        I_2_est_samples[curr_sample_ind] = (var_het - var_without_het) / var_het
    }
    #Set the I^2 estimates for the scenario to the samples computed 
    I_2_est_mat[curr_row_ind, ] = I_2_est_samples
}

In [9]:
#Compute the I^2 estimate for each scenario by taking the median. We replace any values < 0 with 0
#These negative values can occur in scenarios with a small number of studies and/or small samples sizes or those with barely any heterogeneity. 
median_I2 = apply(I_2_est_mat, 1, median)
median_I2[median_I2 < 0] = 0 #Replace negative values with 0 
median_I2

 [1] 0.9420501 0.9018580 0.7190708 0.8632050 0.7520196 0.5647295 0.8732790
 [8] 0.3101344 0.9681097 0.9379354 0.9231907 0.9007681 0.6513969 0.0000000
[15] 0.2446803 0.8542834 0.9244632 0.6787558 0.7173617

In [10]:
scenario_I2 = included_scenarios %>% mutate(I2_est = median_I2) 
output = scenario_df %>% left_join(scenario_I2 %>% select(Scenario, I2_est), by = "Scenario")#Create a DataFrame with scenario labels and the I^2 estimates to be output

In [11]:
saveRDS(output, file.path(output_dir, paste0("I2_Estimates_", sev_class_type, ".rds"))) #Save the output to RDS file for use in visualisations